# Exp 11 - Secure V2X Communication using Encryption and SHA-256 in Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Demonstrate message confidentiality and integrity checks for V2X-style messages.

The notebook uses SHA-256 through HMAC for integrity/authentication demonstration. It also uses a simple XOR transform only to illustrate reversible encoding. XOR is not production-grade encryption.

## Textbook Notes and Case Studies

### 1. Textbook Background

Secure V2X communication must protect message confidentiality, integrity, authenticity, and freshness. Encryption protects content from unauthorized reading. Hashing supports integrity checking. SHA-256 is a cryptographic hash function standardized by NIST as part of the Secure Hash Standard.

Encryption and hashing solve different problems. Encryption does not automatically prove that data was not modified unless an authenticated mode or message authentication method is used. A plain hash detects accidental or visible change only if the receiver has a trusted expected hash. For authentication, systems normally use MACs or digital signatures.

### 2. Architecture Notes

```
Sender Vehicle
  |
  v
Message -> Encrypt / Sign or Hash -> Transmit -> Verify / Decrypt -> Receiver Decision
                                                |
                                                v
                                  Tamper Detection / Accept-Reject
```

The experiment demonstrates message transformation and SHA-256 integrity checking. In a production V2X system, key management, certificates, replay protection, and secure hardware also matter.

### 3. Important Concepts and Formulas

Hash function:

```
digest = SHA256(message)
```

Integrity verification:

```
received_digest == recomputed_digest
```

Confidentiality goal:

```
unauthorized_receiver should not recover plaintext from ciphertext
```

Freshness requirement:

```
timestamp or nonce must prevent reuse of an old valid message
```

### 4. Classroom Case Studies

Case Study A - Hazard Warning Integrity:
A vehicle sends a hazard warning. If an attacker changes "hazard=false" to "hazard=true", a hash or authenticated integrity check should detect tampering.

Case Study B - Location Privacy:
Telemetry containing precise location may need confidentiality. Encryption protects against passive observers, but access control and data minimization are still required.

Case Study C - Emergency Vehicle Signal:
Messages claiming emergency priority must be authenticated. Encryption alone is not enough because the receiver must know whether the sender is authorized.

### 5. Analysis Checklist

Separate confidentiality, integrity, authentication, and freshness in the lab record. Mention which property the notebook demonstrates and which properties require stronger production mechanisms.

### 6. Source Notes

- NIST FIPS 180-4 specifies SHA-256 within the Secure Hash Standard: https://csrc.nist.gov/publications/detail/fips/180/4/final
- Python `hashlib` documentation: https://docs.python.org/3/library/hashlib.html


## Architecture

```text
V2X Message
  |-- vehicle id
  |-- event
  |-- position
  |-- timestamp
          |
          v
Sender Security Layer
  |-- reversible encoding demo
  |-- HMAC-SHA256 tag
          |
          v
Receiver Security Layer
  |-- decode
  |-- recompute HMAC
  |-- compare tags
          |
          v
Accept / Reject
```

## Formulas and Required Theory

HMAC concept:

\[
tag = HMAC_{SHA256}(key, message)
\]

Integrity verification:

\[
\text{valid} = compare\_digest(tag_{received}, HMAC_{SHA256}(key, message_{received}))
\]

If a message changes, the recomputed tag changes. A receiver rejects messages whose tags do not match.

## In-Lab Method

1. Define a byte message and shared key.
2. Generate an HMAC-SHA256 tag.
3. Encode and decode the message.
4. Recompute the tag at the receiver.
5. Use constant-time comparison for verification.

In [1]:
import hashlib
import hmac

print("EXP 11 - IN-LAB SECURE V2X MESSAGE")
key = b"lab-shared-key"
message = b"vehicle=V1;event=hard_brake;x=125;y=48;ts=1002"
digest = hmac.new(key, message, hashlib.sha256).hexdigest()
cipher = bytes(b ^ key[i % len(key)] for i, b in enumerate(message))
plain = bytes(b ^ key[i % len(key)] for i, b in enumerate(cipher))
print("Ciphertext hex:", cipher.hex())
print("SHA-256 HMAC :", digest)
print("Decrypted    :", plain.decode())
print("Integrity ok :", hmac.compare_digest(digest, hmac.new(key, plain, hashlib.sha256).hexdigest()))

EXP 11 - IN-LAB SECURE V2X MESSAGE
Ciphertext hex: 1a040a441004044f3355160e131c02155f45121a052d07164c000042145c531f4653184f515c161f16445d51521f
SHA-256 HMAC : d0e7612d4b2fe8a949d2e3ef0ea38c5db5b0170718412c315b29907b1f09331e
Decrypted    : vehicle=V1;event=hard_brake;x=125;y=48;ts=1002
Integrity ok : True


## Post-Lab Method

The post-lab cell changes the speed field from the original message. The HMAC check fails for the tampered message, proving that modification is detected.

In [2]:
import hashlib
import hmac

print("EXP 11 - POST-LAB TAMPER CHECK")
key = b"lab-shared-key"
original = b"vehicle=V1;speed=45;ts=1002"
tampered = b"vehicle=V1;speed=95;ts=1002"
tag = hmac.new(key, original, hashlib.sha256).hexdigest()
print("Original valid:", hmac.compare_digest(tag, hmac.new(key, original, hashlib.sha256).hexdigest()))
print("Tampered valid:", hmac.compare_digest(tag, hmac.new(key, tampered, hashlib.sha256).hexdigest()))
print("Note: XOR encryption above is only a lab demonstration; production V2X needs vetted cryptography.")

EXP 11 - POST-LAB TAMPER CHECK
Original valid: True
Tampered valid: False
Note: XOR encryption above is only a lab demonstration; production V2X needs vetted cryptography.


## What to Write in the Lab Record

- Include ciphertext hex and HMAC output.
- Explain why hashing alone is not the same as authenticated integrity.
- Explain why `compare_digest` is used.
- Explicitly state that XOR is only a classroom demonstration, not production encryption.

## References

- Python `hmac` documentation: https://docs.python.org/3/library/hmac.html
- Python `hashlib` documentation: https://docs.python.org/3/library/hashlib.html